# Event Summarization using VLM + LLM

**Computer Vision — Assignment 2, Group 6**

This notebook implements a **two-stage event detection pipeline**:
1. **Stage 1 (VLM):** Sample frames uniformly from the video and generate a per-frame description using SmolVLM2
2. **Stage 2 (LLM):** Feed all descriptions to the model in text-only mode to extract a structured list of salient events

The approach handles videos of **any length** — frame sampling and event count are scaled adaptively to video duration.

All four videos (`video_21` – `video_24`) are processed end-to-end with results saved to `outputs/`.

## Section 1 — Setup

Install and import all required libraries.

In [ ]:
!pip install transformers accelerate --quiet
!pip install torch torchvision torchaudio --quiet
!pip install opencv-python pillow pandas num2words av --quiet

In [ ]:
import re
import json
import torch
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from transformers import AutoProcessor, AutoModelForImageTextToText

print("✓ All libraries imported successfully!")

## Section 2 — Configuration

Set videos, model, and pipeline parameters.

In [ ]:
# Videos to process
VIDEOS = ["video_21.mp4", "video_22.mp4", "video_23.mp4", "video_24.mp4"]

# Model
MODEL_NAME = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

# Token budgets
MAX_TOKENS_FRAME = 80    # per-frame description
MAX_TOKENS_EVENT = 512   # event extraction

# Output directory
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Adaptive scaling functions
def compute_num_frames(duration_sec):
    minutes = duration_sec / 60
    if minutes <= 1: return 8
    elif minutes <= 3: return 12
    elif minutes <= 10: return 16
    else: return 24

def compute_target_events(duration_sec):
    minutes = duration_sec / 60
    if minutes <= 2: return 5
    elif minutes <= 5: return 8
    elif minutes <= 10: return 12
    else: return 15

print(f"Model  : {MODEL_NAME}")
print(f"Videos : {VIDEOS}")
print(f"Output : {OUTPUT_DIR}")

## Section 3 — Video Utilities

Functions to inspect videos and sample frames uniformly across the full duration.

In [ ]:
def get_video_info(video_path):
    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur   = total / fps if fps > 0 else 0.0
    cap.release()
    return fps, total, dur

def secs_to_mmss(s):
    m, sec = divmod(int(s), 60)
    return f"{m:02d}:{sec:02d}"

def sample_frames_uniform(video_path, n_frames):
    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0 or fps == 0:
        cap.release()
        return [], []
    
    indices = [int(total * i / n_frames) for i in range(n_frames)]
    frames, timestamps = [], []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
            timestamps.append(round(idx / fps, 2))
    cap.release()
    
    interval = (total / fps) / n_frames
    print(f"  Sampled {len(frames)} frames | ~{interval:.0f}s interval | {secs_to_mmss(timestamps[0])} → {secs_to_mmss(timestamps[-1])}")
    return frames, timestamps

# Preview videos
print("\nVideo Summary:")
for v in VIDEOS:
    if Path(v).exists():
        fps, tot, dur = get_video_info(v)
        print(f"  {v}: {dur:.0f}s ({dur/60:.1f}m) | {compute_num_frames(dur)} frames | {compute_target_events(dur)} events")
    else:
        print(f"  {v}: NOT FOUND")

## Section 4 — Load the Model

Load SmolVLM2 with automatic device detection (GPU or CPU).

In [ ]:
# Detect device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print(f"\nLoading '{MODEL_NAME}' on {DEVICE} ({DTYPE})...")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
).to(DEVICE)

print(f"✓ Model loaded successfully on {DEVICE}!")

## Section 5 — Stage 1: VLM Per-Frame Descriptions

Run SmolVLM2 on each frame individually to generate per-frame descriptions.

In [ ]:
FRAME_DESC_PROMPT = (
    "Describe only what is happening in this single video frame. "
    "Focus on people, their actions, and the scene context. "
    "Be concise: at most 2 sentences, under 40 words."
)

def describe_frames(frames, timestamps):
    results = []
    for frame, ts_sec in zip(frames, timestamps):
        ts_label = secs_to_mmss(ts_sec)
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "url": frame},
                {"type": "text",  "text": FRAME_DESC_PROMPT},
            ],
        }]
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(DEVICE, dtype=DTYPE)

        with torch.no_grad():
            ids = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_TOKENS_FRAME)

        raw  = processor.batch_decode(ids, skip_special_tokens=True)[0]
        desc = raw.split("Assistant:")[-1].strip() if "Assistant:" in raw else raw.strip()
        results.append((ts_label, desc))
        print(f"  [{ts_label}] {desc[:100]}")

    return results

print("✓ Stage 1 (VLM frame descriptions) ready.")

## Section 6 — Stage 2: LLM Event Extraction

Feed all frame descriptions to the model in text-only mode to extract structured events.

In [ ]:
def build_llm_prompt(frame_descriptions, duration_sec):
    n_events   = compute_target_events(duration_sec)
    dur_str    = f"{duration_sec / 60:.1f} minutes"
    desc_block = "\n".join(f"[{ts}]: {desc}" for ts, desc in frame_descriptions)
    ts_list    = ", ".join(ts for ts, _ in frame_descriptions)

    return (
        f"You are given frame-by-frame descriptions of a {dur_str} video:\n\n"
        f"{desc_block}\n\n"
        f"Your task: extract exactly {n_events} salient events from these descriptions.\n\n"
        f"RULES:\n"
        f"- A salient event is a meaningful change, action, or transition.\n"
        f"- Group consecutive frames with similar content into ONE event.\n"
        f"- NEVER repeat the same action type (e.g. 'person walks' appears at most once).\n"
        f"- Every event must be fundamentally different from all others.\n"
        f"- Use MM:SS - MM:SS timestamp ranges using ONLY values from: {ts_list}\n\n"
        f"OUTPUT FORMAT — strictly one event per line:\n"
        f"Event 1: <description>, MM:SS - MM:SS\n"
        f"Event 2: <description>, MM:SS - MM:SS\n"
        f"...\n"
        f"Event {n_events}: <description>, MM:SS - MM:SS\n\n"
        f"Generate EXACTLY {n_events} events, no more, no less."
    )

def extract_events_with_llm(frame_descriptions, duration_sec):
    prompt   = build_llm_prompt(frame_descriptions, duration_sec)
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs   = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(DEVICE, dtype=DTYPE)

    with torch.no_grad():
        ids = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_TOKENS_EVENT)

    raw = processor.batch_decode(ids, skip_special_tokens=True)[0]
    return raw.split("Assistant:")[-1].strip() if "Assistant:" in raw else raw.strip()

print("✓ Stage 2 (LLM event extraction) ready.")

## Section 7 — Output Parser

Convert raw LLM text into a structured pandas DataFrame.

In [ ]:
def parse_events(raw_text):
    ts_re    = r"(\d{1,2}:\d{2})\s*[-–]\s*(\d{1,2}:\d{2})"
    pat_ev   = re.compile(r"Event\s+\d+\s*:\s*(.+?),\s*" + ts_re, re.IGNORECASE)
    pat_num  = re.compile(r"^\d+\.\s+(.+?),\s*" + ts_re)

    rows = []
    for line in raw_text.splitlines():
        m = pat_ev.search(line.strip())
        if m:
            rows.append((m.group(1).strip(), m.group(2), m.group(3)))
    if not rows:
        for line in raw_text.splitlines():
            m = pat_num.match(line.strip())
            if m:
                rows.append((m.group(1).strip(), m.group(2), m.group(3)))
    return rows

def _ts_to_sec(ts):
    try:
        p = [int(x) for x in ts.split(":")]
        return p[0] * 60 + p[1] if len(p) == 2 else p[0] * 3600 + p[1] * 60 + p[2]
    except Exception:
        return None

def build_dataframe(rows, video_name):
    return pd.DataFrame([
        {
            "event_number":  i,
            "video":         video_name,
            "description":   desc,
            "start_time":    start,
            "end_time":      end,
            "start_seconds": _ts_to_sec(start),
            "end_seconds":   _ts_to_sec(end),
        }
        for i, (desc, start, end) in enumerate(rows, 1)
    ])

print("✓ Parser ready.")

## Section 8 — Full Pipeline

`process_video()` ties both stages together.

In [ ]:
def process_video(video_path):
    video_path = str(video_path)
    _, _, duration = get_video_info(video_path)
    n_frames   = compute_num_frames(duration)
    n_events   = compute_target_events(duration)

    print(f"\n{'='*65}")
    print(f"VIDEO : {video_path}")
    print(f"  Duration : {duration:.1f}s ({duration/60:.1f} min)")
    print(f"  Frames   : {n_frames} | Target events: {n_events}")
    print(f"{'='*65}")

    print("\nStage 1 — VLM per-frame descriptions:")
    frames, timestamps = sample_frames_uniform(video_path, n_frames)
    frame_descs = describe_frames(frames, timestamps)

    print("\nStage 2 — LLM event extraction (text-only):")
    raw_output = extract_events_with_llm(frame_descs, duration)
    print(raw_output)

    rows = parse_events(raw_output)
    if rows:
        df = build_dataframe(rows, Path(video_path).name)
    else:
        print("WARNING: no events parsed.")
        df = pd.DataFrame(columns=["event_number", "video", "description", "start_time", "end_time", "start_seconds", "end_seconds"])

    print(f"\n→ {len(df)} events extracted")
    return df, frame_descs, raw_output

print("✓ process_video() ready.")

## Section 9 — Run Pipeline on All Videos

In [ ]:
all_results = {}

for video in VIDEOS:
    if not Path(video).exists():
        print(f"SKIP: {video} not found.")
        continue
    df, descs, raw = process_video(video)
    all_results[video] = {
        "df":                 df,
        "frame_descriptions": descs,
        "raw_llm_output":     raw,
    }

print("\n" + "="*65)
print("All videos processed:")
for v, r in all_results.items():
    print(f"  {v}: {len(r['df'])} events")

## Section 10 — Save Outputs

In [ ]:
for video, res in all_results.items():
    stem  = Path(video).stem
    df    = res["df"]
    descs = res["frame_descriptions"]
    raw   = res["raw_llm_output"]

    txt_path = OUTPUT_DIR / f"{stem}_events.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"Video: {video}\n{'='*60}\n")
        f.write("STAGE 1 — Frame Descriptions:\n")
        for ts, desc in descs:
            f.write(f"  [{ts}] {desc}\n")
        f.write(f"\nSTAGE 2 — Raw LLM Output:\n{raw}\n")

    csv_path = OUTPUT_DIR / f"{stem}_events.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved: {txt_path} | {csv_path}")

all_dfs = [r["df"] for r in all_results.values() if len(r["df"]) > 0]
if all_dfs:
    pd.concat(all_dfs, ignore_index=True).to_csv(
        OUTPUT_DIR / "all_videos_events.csv", index=False
    )
    print(f"Combined CSV: {OUTPUT_DIR / 'all_videos_events.csv'}")

json_payload = {}
for video, res in all_results.items():
    _, _, dur = get_video_info(video)
    json_payload[video] = {
        "duration_seconds":   round(dur, 2),
        "frames_sampled":     len(res["frame_descriptions"]),
        "target_events":      compute_target_events(dur),
        "events_found":       len(res["df"]),
        "frame_descriptions": [{"timestamp": ts, "description": d} for ts, d in res["frame_descriptions"]],
        "raw_llm_output": res["raw_llm_output"],
        "events": res["df"].to_dict(orient="records"),
    }

json_path = OUTPUT_DIR / "events.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_payload, f, indent=2, ensure_ascii=False)
print(f"JSON: {json_path}")
print(f"\n✓ All outputs saved to {OUTPUT_DIR}/")

## Section 11 — Display Results

In [ ]:
for video, res in all_results.items():
    df = res["df"]
    print(f"\n{'='*65}")
    print(f"VIDEO: {video} ({len(df)} events)")
    print(f"{'='*65}")
    if len(df) > 0:
        display(df[["event_number", "start_time", "end_time", "description"]])
    else:
        print("  No events parsed.")